# Module 2 — Using Claude Code for String + File Processing

**SBML Lab Intern Training**

---

This module shows how Claude Code can generate file-processing and data-manipulation scripts from natural language descriptions.

The core skill is simple:

> **Describe what you want in plain English → get a script → refine it with follow-ups.**

By the end of this module you will have practiced this loop on GFF file parsing and pandas workflows — two tasks you will encounter constantly in the lab.


## Background: Genome Annotations and the GFF Format

### What is a genome annotation?

A genome is just a long string of DNA — billions of base pairs with no obvious structure by itself. A **genome annotation** is a map that labels where things are: this region is a gene, this region encodes a protein. Without an annotation, you have sequence but no biology.

The lab's annotation for *E. coli* K-12 MG1655 marks every gene in the genome — about 4,600 features total (all rows are `gene` features; this file has no separate `CDS` rows). This file is the reference you'll work with throughout the training.

### The GFF format

GFF (General Feature Format) is the standard file format for genome annotations. Every row describes one genomic feature (a gene, CDS, etc.) using 9 tab-separated columns:

The columns below describe the GFF format in general; the **Example** column shows the actual values found in this lab annotation file:

| Column | Name | Example (this file) |
|--------|------|---------|
| 1 | Chromosome / sequence name | `NC_000913` |
| 2 | Source | `Ecocyc_14.1` |
| 3 | Feature type | `gene` |
| 4 | Start position | `337` |
| 5 | End position | `2799` |
| 6 | Score | `.` (unused here) |
| 7 | Strand | `+` or `-` |
| 8 | Phase | `.` |
| 9 | Attributes | `locus_tag=b0002;gene=thrA;product=...;color=00FF00` |

> **Note the attribute keys:** this file uses `locus_tag=` and `gene=` (not `ID=`/`Name=`). Parse for those keys when extracting gene names.
>
> **Sequence-name convention:** this annotation uses `NC_000913` (no version suffix). The reference genome you download in Module 3 is `NC_000913.3`. The suffix differs — this matters when you join files by chromosome (Module 4), where `NC_000913` and `NC_000913.3` will not match unless you reconcile them.

**Coordinates are 1-based** — position 1 is the first nucleotide of the chromosome. This matters when writing Python code, since Python string indexing is 0-based. An off-by-one error here silently shifts every position by one.

### Why does this lab use GFF?

GFF is the input format for **MetaScope**, the lab's genome browser for visualizing ChIP-exo data. The alignment pipeline (Module 3) produces a GFF file as its final output so reads can be loaded into MetaScope alongside the gene annotation.

---

## GFF Format Recap

GFF is the standard annotation format used in this lab — tab-delimited, 9 columns, no header row, 1-based coordinates. Use `lab-context.md` as a reference if needed.

**Data for this module:** the lab *E. coli* K-12 MG1655 annotation GFF is pre-provided at:
```
data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
```
This is the same file used in Modules 4 and 5 (the mini-project).


## Natural Language → Script

The pattern you will use throughout this module:

1. **Write a plain-English description** of what you want the script to do. Be specific: mention the file format, the columns involved, and the expected output.
2. **Claude Code generates a first draft.** Paste it into a code cell and run it.
3. **Follow up with refinements.** Fix edge cases, add output formatting, handle errors, change the logic.

> **Important:** your first prompt is a draft, not a final answer. Follow-up prompts are where the real work happens. Most production scripts in the lab went through 3–10 iterations before they were committed.

### Tips for better first prompts

- Name the file format explicitly: *"a GFF file"*, not *"a text file"*.
- State the output: *"print the result to stdout"* or *"save to a new TSV file"*.
- Mention what to do with edge cases: *"skip comment lines"*, *"ignore rows where score is `.`"*.

---

### Why would you filter a genome annotation?

Real analyses rarely use the entire annotation. Common filtering scenarios in this lab:

- **By strand:** ChIP-exo reads are strand-specific — a binding site on the `+` strand regulates genes on the `+` strand. You often want to look at only one strand at a time.
- **By feature type:** You might want only `gene` entries, not `CDS`, depending on the question.
- **By coordinate range:** If you're zooming in on a genomic region (say, the first 500 kb), you only want features within those coordinates.

### What does "strand" mean?

The *E. coli* chromosome is double-stranded DNA. Genes can be encoded on either strand:

- **`+` strand (forward):** The gene is read left-to-right as written in the reference FASTA. Start < End.
- **`-` strand (reverse):** The gene is read right-to-left. In GFF, start is still less than end (by convention), but the strand column is `-`.

This matters biologically because a TF binding site upstream of a `+` strand gene is in a different genomic direction than one upstream of a `-` strand gene.

---

### Exercise 1 — Generate and refine a GFF filtering script

1. Describe a GFF filtering task in plain English to Claude Code. Use the lab annotation GFF at `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`. For example: filter by feature type, by strand, or by coordinate range — your choice.
2. Generate the script.
3. Improve it with **at least two follow-up prompts**.

Write the **final version** of your script in the code cell below. Add a comment at the top of the script that briefly describes what each follow-up prompt changed.

Example comment format:
```python
# Follow-up 1: added argparse so the input path is a CLI argument
# Follow-up 2: changed output to TSV instead of printing to stdout
```

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [5]:
# Your final script here
# Follow-up 1: added strand as a CLI argument so '+' or '-' can be chosen (instead of hardcoded '+')
# Follow-up 2: added total/kept row counts and percentage printed after filtering

def filter_by_strand(in_path, out_path, strand):
    total = 0
    kept = 0
    with open(in_path) as infile, open(out_path, "w") as outfile:
        for line in infile:
            total += 1
            fields = line.rstrip("\n").split("\t")
            if fields[6] == strand:
                kept += 1
                outfile.write(line)
    return total, kept

# 노트북 안에서는 sys.argv 대신 직접 함수를 호출
in_path = "../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff"
out_path = "filtered_plus_strand.gff"
strand = "+"

total, kept = filter_by_strand(in_path, out_path, strand)
print(f"{kept}/{total} rows kept ({kept / total:.1%})")

2267/4595 rows kept (49.3%)


> **Explain it**
>
>이 스크립트는 GFF 파일을 한 줄씩 읽어 탭으로 분리한 뒤, 7번째 컬럼(인덱스 6, strand)이 
지정한 값과 일치하는 행만 새 파일에 쓴다. Claude는 strand 값을 하드코딩하지 않고 함수 
파라미터로 받도록 설계해 '+'와 '-' 양쪽에 재사용 가능하게 만들었고, 전체 행 수(total)와 
필터링 후 남은 행 수(kept)를 함께 반환해 필터링 결과를 검증하기 쉽게 했다.

In [9]:
from filter_gff_by_strand import filter_gff  # 또는 %run으로 파일 실행 후 함수 재사용

# 예시 1: strand만
total, kept = filter_gff("../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff",
                          "test1.gff", strand="+")
print(f"strand만: {kept}/{total}")

# 예시 2: featuretype만
total, kept = filter_gff("../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff",
                          "test2.gff", featuretype="gene")
print(f"featuretype만: {kept}/{total}")

# 예시 3: strand + positionrange 동시에
total, kept = filter_gff("../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff",
                          "test3.gff", strand="+", positionrange=(500000, 1000000))
print(f"strand+범위: {kept}/{total}")

# 존재하지 않는 feature type으로 테스트 (0이 나와야 정상)
total, kept = filter_gff("../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff",
                          "test4.gff", featuretype="CDS")
print(f"CDS 필터 (없어야 함): {kept}/{total}")

# 범위만 (strand 없이)
total, kept = filter_gff("../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff",
                          "test5.gff", positionrange=(500000, 1000000))
print(f"범위만: {kept}/{total}")

# Bonus extension: added --featuretype and --positionrange as optional filters,
   # alongside strand. All three combine with AND logic and can be used independently.

strand만: 2267/4595
featuretype만: 4595/4595
strand+범위: 251/4595
CDS 필터 (없어야 함): 0/4595
범위만: 477/4595


>**Explain it**
>
> 원래는 filter_gff_by_strand.py를 strand로만 필터링 하였는데, 실제 분석에서는 feature type이나, 좌표 범위로도 필터링 해야 하는 경우가 많기에 이 기능이 부족하다고 느껴 --featuretype과 --positionrange 옵션을 추가해 봄. 결과, 세 조건(strand, featuretype, positionrange)을 각각 독립적으로 켜고 끌 수 있으면서도 동시에 조합(AND 조건)해서 쓸 수 있는 필터링 스크립트가 완성됨. 검증을 위해 존재하지 않는 featuretype(CDS)으로 테스트해 0/4595가 나오는 것으로 필터가 정확히 작동함을 확인했고, positionrange 단독 결과 (477개)가 strand+positionrange 조합 결과(251개)보다 크다는 것을 통해 조건들이 AND로 누적 적용됨을 검증함.

## Loading GFF Files with pandas

Loading a GFF file with pandas requires a few non-obvious options. Use Claude Code to figure out the correct `read_csv` call for `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`.

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [9]:
import pandas as pd

gff = pd.read_csv(
    "../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff",
    sep="\t",
    header=None,
    names=["seqid", "source", "feature", "start", "end", "score", "strand", "phase", "attributes"]
)

gff.head()

,seqid,source,feature,start,end,score,strand,phase,attributes
0,NC_000913,Ecocyc_14.1,gene,190,255,.,+,.,locus_tag=b0001;gene=thrL;feature=gene;product...
1,NC_000913,Ecocyc_14.1,gene,337,2799,.,+,.,locus_tag=b0002;gene=thrA;feature=gene;product...
2,NC_000913,Ecocyc_14.1,gene,2801,3733,.,+,.,locus_tag=b0003;gene=thrB;feature=gene;product...
3,NC_000913,Ecocyc_14.1,gene,3734,5020,.,+,.,locus_tag=b0004;gene=thrC;feature=gene;product...
4,NC_000913,Ecocyc_14.1,gene,5234,5530,.,+,.,locus_tag=b0005;gene=yaaX;feature=gene;product...


### Exercise 2 — Debugging: Loading and Indexing

The script below has **two bugs**. Use Claude Code to diagnose and fix them.

Workflow:
1. Read the script and try to spot the bugs yourself first.
2. Paste the script into Claude Code and ask it to find the bugs.
3. Use `/debug` in the Claude Code terminal if you get stuck.
4. Write the corrected version in the empty code cell below the broken one.

> **Hint:** there is one bug in how the file is loaded, and one bug in how a row is accessed.

> **Explain it:** after fixing it, write 1–2 sentences on *what each bug was and why your fix works*. Understanding the bug matters more than the patch.

In [2]:
import pandas as pd

# Load GFF file
df = pd.read_csv('data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff', sep='\t')

# Filter for forward strand genes
forward = df[df['strand'] == '+']

# Get the 5th feature
fifth = forward.iat[5]

print(f"5th forward feature starts at position {fifth['start']}")
     

FileNotFoundError: [Errno 2] No such file or directory: 'data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff'

In [5]:
# Fixed version here

import pandas as pd

# Load GFF file
df = pd.read_csv(
    '../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff',
    sep='\t',
    header=None,
    names=["seqid", "source", "feature", "start", "end", "score", "strand", "phase", "attributes"]
)

# Filter for forward strand genes
forward = df[df['strand'] == '+']

# Get the 5th feature
fifth = forward.iloc[4]
print(f"5th forward feature starts at position {fifth['start']}")

5th forward feature starts at position 5234


> **Explain it**
>
> 첫 번째 버그는 read_csv에 header=None과 names=[...]가 빠져 있어서 pandas가 첫 데이터 행을 컬럼명으로 잘못 인식해 strand 컬럼을 찾지 못한 것이었다. 
>
> 두 번째 버그는 .iat[5]가 행과 열을 모두 지정해야 하는 단일 값 접근자인데 인자를 하나만 줘서 발생했으며, 행 전체를 가져오는 .iloc로 바꿔야 했다. 또한 pandas의 iloc는 0부터 세므로 "5번째" 행은 iloc[4]로 접근해야 정확하다.

## `iterrows()` — When and Why

`iterrows()` iterates over a DataFrame row by row. Use Claude Code to understand when it is appropriate and when to avoid it.

### Exercise 3a — When to use it (concept)

After getting Claude Code's explanation, write a **2-sentence answer** in the markdown cell below:
- Sentence 1: when you **should** use `iterrows()`.
- Sentence 2: when you **should avoid** it and what to use instead.

### Exercise 3b — Use it on real data (hands-on)

Now actually iterate. Using the annotation DataFrame you loaded above, **find the gene whose transcription start site (TSS) is closest to position 1,000,000** on the genome.

- For a `+`-strand gene the TSS is its `start`; for a `-`-strand gene the TSS is its `end`.
- Loop over rows with `df.iterrows()`, compute each gene's distance to 1,000,000, and report the closest gene's name (the `gene=` value in the attribute column) and its distance.

Write the working script in the code cell below. When it runs, ask Claude Code how you would do the *same* computation **without** `iterrows()` (vectorized) — this is the practical version of the concept from Exercise 2a.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [9]:
# Exercise 3b — find the gene whose TSS is closest to position 1,000,000, using df.iterrows()
# Your script here.

import re

target_pos = 1_000_000

closest_idx = None
closest_distance = None

for idx, row in df.iterrows():  # df로 수정
    tss = row['start'] if row['strand'] == '+' else row['end']
    distance = abs(tss - target_pos)

    if closest_distance is None or distance < closest_distance:
        closest_distance = distance
        closest_idx = idx

closest_row = df.loc[closest_idx]  # 여기도 df로 수정
match = re.search(r'gene=([^;]+)', closest_row['attributes'])
gene_name = match.group(1) if match else 'Unknown'

print(f"가장 가까운 유전자: {gene_name}, 거리: {closest_distance} bp")

가장 가까운 유전자: ycbT, 거리: 1030 bp


In [10]:
import numpy as np
import re

target_pos = 1_000_000

tss = np.where(df['strand'] == '+', df['start'], df['end'])
distance = pd.Series(np.abs(tss - target_pos), index=df.index)

closest_idx = distance.idxmin()
closest_distance = distance[closest_idx]

match = re.search(r'gene=([^;]+)', df.loc[closest_idx, 'attributes'])
gene_name = match.group(1) if match else 'Unknown'

print(f"가장 가까운 유전자: {gene_name}, 거리: {closest_distance} bp")

가장 가까운 유전자: ycbT, 거리: 1030 bp


> **Answer**
>
> iterrows()는 행 수가 적거나(수십~수백 행) 디버깅처럼 한 번 훑어보는 용도, 또는 각 행마다 외부 API 호출처럼 반복문이 꼭 필요한 부수효과가 있을 때 사용해야 한다.
>
> 반면 대량의 GFF/BED 데이터를 필터링하거나 컬럼 간 산술 연산, 조건별 값 계산처럼 "숫자 계산"이 주가 되는 작업에서는 iterrows()를 피하고, boolean masking이나 df['end'] - df['start']와 같은 벡터화 연산을 사용해야 훨씬 빠르다.


## End of Session

Before closing, run `/log` in the Claude Code terminal to save a summary of your session.

This creates a record of the prompts you used and the scripts you generated — useful for your own reference and for lab documentation.

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module2): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
